# LLM discovery and communal knowledge

Agents explicitly choose **deposit** and **retrieve** operations. Compare access to their own deposited facts with access to peers' facts, holding the population profile and learning architecture fixed.

The default runs real local SmolLM2 with private LoRA. Training is disabled below so **Run All** first shows the plan and any saved results. Read [the experiment design](../docs/discovery.md) for costs, controls and interpretation. Run only one full CPU experiment at a time.

In [ ]:
from pathlib import Path
import os
import json
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/discovery.yaml").exists(), "Open this notebook from the millstlabs repository."
os.chdir(ROOT)
from millstlabs.config import load_config
from millstlabs.discovery import proposal, report

RUN = False                 # Set True to start/resume; this cell alone never starts training.
MODE = "run"                # "smoke" runs 128 decisions/arm using the real LLM.
HOURS = 12
FULL_FACTORIAL = False      # Eight arms instead of four; doubles the budget.
OUTPUT = ROOT / ("runs/discovery-smoke" if MODE == "smoke" else
                 "runs/discovery-factorial" if FULL_FACTORIAL else "runs/discovery-proof")
print("Python kernel:", sys.executable)
print("Output:", OUTPUT)

In [ ]:
cfg = load_config(ROOT / "configs/discovery.yaml")
print(json.dumps(proposal(cfg, FULL_FACTORIAL), indent=2))
if MODE == "smoke":
    print("SMOKE overrides this full plan to 128 decisions/arm, 64 demonstrations and an 8-tick evaluation.")

## Launch or resume

The next cell blocks while the runner works, displaying progress. A 12-hour soft limit may require another invocation; repeating the same settings resumes. Output locks reject a second runner on the same deployment. A Mac terminal can keep the machine awake with:

```bash
caffeinate -i bash scripts/discovery.sh --mode run --hours 12
```

Do not interpret the smoke test's eight-tick survival as task competence.

In [ ]:
if RUN:
    command = ["bash", "scripts/discovery.sh", "--mode", MODE, "--hours", str(HOURS), "--output", str(OUTPUT)]
    if FULL_FACTORIAL:
        command.append("--full")
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Training disabled. Set RUN=True and execute this cell to launch/resume.")

In [ ]:
if (OUTPUT / "deployment.json").exists():
    saved_cfg = load_config(OUTPUT / "config.yaml")
    saved_plan = json.loads((OUTPUT / "deployment.json").read_text())
    result = report(saved_cfg, OUTPUT, len(saved_plan["conditions"]) == 8)
else:
    result = report(cfg, OUTPUT, FULL_FACTORIAL)
print(f"Complete: {result['complete']}/{result['total']}")
for row in result["runs"]:
    print(row["condition"], "seed", row["seed"], row["status"], "decisions", row.get("decisions", 0))
for comparison in result["comparisons"]:
    print(comparison["variant"], json.dumps({"relative_discovery_gain": comparison["relative_discovery_gain"],
          "checks": comparison["checks"], "exploratory_poc_pass": comparison["exploratory_poc_pass"]}, indent=2))
print(result["note"])

In [ ]:
import matplotlib.pyplot as plt
finished = [r for r in result["runs"] if r.get("final_metrics")]
if finished:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, metric, label in zip(axes,
            ["discoveries_per_1000_opportunities", "restricted_mean_lifetime", "food_per_alive_decision"],
            ["New facts / 1,000 allocated opportunities", "Mean lifetime (ticks)", "Food / living decision"]):
        ax.bar([r["condition"] for r in finished], [r["final_metrics"][metric] for r in finished])
        ax.set_title(label)
        ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    plt.show()
else:
    print("No completed runs yet; rerun these report cells after training.")

## Interpret the result

Discovery is the union of **first-hand observations**, excluding the initial view, in both arms. Copying a peer's fact cannot increase it. The fixed denominator includes opportunities lost after death. Read `evaluation.jsonl` for map-level discovery curves and timing, and `corpus_events.jsonl` for actual deposits and reads.

A corpus benefit is distinct from learning through RL: inspect `change_from_warmstart` and `change_in_corpus_advantage` in `discovery-report.json`. The final `evaluation_private_corpus.jsonl` tests the same shared policy with peer access removed. A one-seed positive result supports a narrow feasibility claim; it does not establish general swarm improvement. Look at food and survival alongside exploration.